In [2]:
import os
import numpy as np
import h5py
from scipy import signal
from scipy.stats import pearsonr
import seaborn as sns
import datetime
import pandas as pd
import matplotlib.pyplot as plt
import string

# Define path to data:
script_dir = os.path.dirname(os.path.abspath("Figure 1 Data Summary.ipynb"))
parent_dir = os.path.dirname(script_dir)
data_path = (parent_dir + '/dfs/')

In [3]:
# Load diary data file:
df_diary = pd.read_csv(data_path+'df_diary_git.csv')


print(len(df_diary))

2630


In [45]:
# Convert 'SO' and 'FA' to datetime and compute time in minutes from MidNight of day before FA:
df_diary['SO'] = pd.to_datetime(df_diary['SO'], format='%I:%M %p')
df_diary['SO_FromMidNight'] = df_diary['SO'].dt.hour + df_diary['SO'].dt.minute/60

df_diary['FA'] = pd.to_datetime(df_diary['FA'], format='%I:%M %p')
df_diary['FA_FromMidNight'] = df_diary['FA'].dt.hour + df_diary['FA'].dt.minute/60

# Create a boolean variable testing wether SO and FA were on the same date (child went to sleep after midnight)
condition = df_diary['SO'].dt.date == df_diary['FA'].dt.date
# When true add 24 hours to SO
df_diary.loc[condition, 'SO_FromMidNight'] = df_diary.loc[condition, 'SO_FromMidNight'] + 24
# Always add 24 hours to FA
df_diary['FA_FromMidNight'] = df_diary['FA_FromMidNight'] + 24

# Clean the data:
# Keep all nights where SO was after 6pm on day preceding FA and before 7am on day of FA
df_diary_clean = df_diary[df_diary['SO_FromMidNight'] > 18]
df_diary_clean = df_diary_clean[df_diary_clean['SO_FromMidNight'] < 31]

# Keep all nights where TST > 3 hours and TST < 14 hours
df_diary_clean = df_diary_clean[df_diary_clean['TST'] < 960]
df_diary_clean = df_diary_clean[df_diary_clean['TST'] > 180]

# Keep all nights with WASO < 3 hours
df_diary_clean = df_diary_clean[df_diary_clean['WASO'] < 180]

# Keep all nights with FA before 2pm on day of FA
df_diary_clean = df_diary_clean[df_diary_clean['FA_FromMidNight'] < 38]